In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

def load_and_prepare():
    sci = pd.read_csv("dataset_scientific.csv");  sci["SK"] = 1
    non = pd.read_csv("dataset_non_scientific.csv");  non["SK"] = 0
    df = pd.concat([sci, non], ignore_index=True)

    repo_col = "repo" if "repo" in df.columns else ("Repository" if "Repository" in df.columns else None)
    if repo_col is None:
        raise ValueError("No repo column found.")
    canon = {"trilinos":"Trilinos","amrex":"AMReX","mantid":"Mantid"}
    df["Repo"] = df[repo_col].astype(str).str.strip().str.lower().map(canon)

    # Files_Changed
    if "num_of_files_changed" in df.columns:
        df["Files_Changed"] = pd.to_numeric(df["num_of_files_changed"], errors="coerce")
    elif "files_changed" in df.columns:
        df["Files_Changed"] = df["files_changed"].apply(
            lambda s: 0 if pd.isna(s) else len([p for p in str(s).split(";") if p.strip()])
        )
    else:
        df["Files_Changed"] = pd.NA

    # Lines_Total
    if "total_lines_changed" in df.columns:
        df["Lines_Total"] = pd.to_numeric(df["total_lines_changed"], errors="coerce")
    elif {"lines_added","lines_deleted"}.issubset(df.columns):
        df["Lines_Total"] = pd.to_numeric(df["lines_added"], errors="coerce") + pd.to_numeric(df["lines_deleted"], errors="coerce")
    elif "lines_added" in df.columns:
        df["Lines_Total"] = pd.to_numeric(df["lines_added"], errors="coerce")
    else:
        df["Lines_Total"] = pd.NA

    # Reviewers
    if "unique_reviewers" in df.columns:
        df["Reviewers"] = pd.to_numeric(df["unique_reviewers"], errors="coerce")
    elif "number_of_unique_reviewers" in df.columns:
        df["Reviewers"] = pd.to_numeric(df["number_of_unique_reviewers"], errors="coerce")
    else:
        df["Reviewers"] = pd.NA

    return df

def choose_baseline(m):
    if (m["Repo"]=="AMReX").any(): return "AMReX"
    if (m["Repo"]=="Trilinos").any(): return "Trilinos"
    if (m["Repo"]=="Mantid").any(): return "Mantid"
    return m["Repo"].mode().iat[0]

def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""


## Time to Merge

In [2]:
import pandas as pd
import statsmodels.formula.api as smf

df = load_and_prepare()

# Convert dependent variable from seconds → days
df["time_to_merge_days"] = pd.to_numeric(df["time_to_merge"], errors="coerce") / 86400.0
Y = "time_to_merge_days"

need = [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers", "Repo"]
m = df[need].copy()

# Ensure numeric conversion for regressors
for c in [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")

# Drop missing values
m = m.dropna(subset=need)

# Baseline repo
baseline = choose_baseline(m)

# Regression model
formula = f"{Y} ~ SK + Files_Changed + Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
ols = smf.ols(formula=formula, data=m).fit(cov_type="HC3")

print("\n[Time to Merge — in DAYS]")
print(f"N={int(ols.nobs)}, R^2={ols.rsquared:.3f}")
print(f"SK: {ols.params['SK']:.2f} days (p={ols.pvalues['SK']:.4f}) {stars(ols.pvalues['SK'])}")
for term in ["Files_Changed","Lines_Total","Reviewers"]:
    if term in ols.params:
        print(f"{term}: {ols.params[term]:.2f} days (p={ols.pvalues[term]:.4f}) {stars(ols.pvalues[term])}")



[Time to Merge — in DAYS]
N=2274, R^2=0.075
SK: 7.58 days (p=0.0000) ***
Files_Changed: 0.39 days (p=0.0004) ***
Lines_Total: 0.00 days (p=0.6760) 
Reviewers: 5.59 days (p=0.0002) ***


In [3]:
# log model
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = load_and_prepare()

# Convert dependent variable from seconds → days
df["time_to_merge_days"] = pd.to_numeric(df["time_to_merge"], errors="coerce") / 86400.0

# Log-transform (safe for zeros)
df["log_time_to_merge"] = np.log1p(df["time_to_merge_days"])
df["log_Files_Changed"] = np.log1p(pd.to_numeric(df["Files_Changed"], errors="coerce"))
df["log_Lines_Total"]   = np.log1p(pd.to_numeric(df["Lines_Total"], errors="coerce"))
df["Reviewers"]         = pd.to_numeric(df["Reviewers"], errors="coerce")

# Prepare data
Y = "log_time_to_merge"
need = [Y, "SK", "log_Files_Changed", "log_Lines_Total", "Reviewers", "Repo"]
m = df[need].dropna()

# Choose baseline repo
baseline = choose_baseline(m)

# Regression model (log-log form)
formula = f"{Y} ~ SK + log_Files_Changed + log_Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
ols = smf.ols(formula=formula, data=m).fit(cov_type="HC3")

def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""

print("\n[Log-Transformed Time to Merge]")
print(f"N={int(ols.nobs)}, R²={ols.rsquared:.3f}")
for term in ["SK", "log_Files_Changed", "log_Lines_Total", "Reviewers"]:
    if term in ols.params:
        print(f"{term}: {ols.params[term]:.3f} (p={ols.pvalues[term]:.4f}) {stars(ols.pvalues[term])}")

print("\n[Interpretation in % Change of Merge Time]")
for term in ["SK", "log_Files_Changed", "log_Lines_Total", "Reviewers"]:
    if term in ols.params:
        b, p = ols.params[term], ols.pvalues[term]
        pct = (np.expm1(b)) * 100
        print(f"{term}: {pct:+.1f}% change in merge time (p={p:.4f}) {stars(p)}")



[Log-Transformed Time to Merge]
N=2274, R²=0.301
SK: 0.512 (p=0.0000) ***
log_Files_Changed: 0.304 (p=0.0000) ***
log_Lines_Total: 0.011 (p=0.3849) 
Reviewers: 0.377 (p=0.0000) ***

[Interpretation in % Change of Merge Time]
SK: +66.9% change in merge time (p=0.0000) ***
log_Files_Changed: +35.6% change in merge time (p=0.0000) ***
log_Lines_Total: +1.1% change in merge time (p=0.3849) 
Reviewers: +45.7% change in merge time (p=0.0000) ***


## Unique Reviewers

In [4]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = load_and_prepare()

Y = "unique_reviewers" if "unique_reviewers" in df.columns else "number_of_unique_reviewers"

need = [Y, "SK", "Files_Changed", "Lines_Total", "Repo"]
m = df[need].copy()

for c in [Y, "SK", "Files_Changed", "Lines_Total"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")

m = m.dropna(subset=need)

baseline = choose_baseline(m)

formula = f"{Y} ~ SK + Files_Changed + Lines_Total + C(Repo, Treatment(reference='{baseline}'))"
nb = smf.glm(formula=formula, data=m, family=sm.families.NegativeBinomial()).fit(cov_type="HC3")

print("\n[Unique Reviewers]")
print(f"N={int(nb.nobs)}, Pseudo-R^2={getattr(nb, 'prsquared', float('nan')):.3f}")
print(f"SK: {nb.params['SK']:.3f} (p={nb.pvalues['SK']:.4f}) {stars(nb.pvalues['SK'])}")
for term in ["Files_Changed", "Lines_Total"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")



[Unique Reviewers]
N=2274, Pseudo-R^2=nan
SK: 0.516 (p=0.0000) ***
Files_Changed: 0.007 (p=0.0000) ***
Lines_Total: 0.000 (p=0.1383) 


/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [5]:
# log
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = load_and_prepare()

Y = "unique_reviewers" if "unique_reviewers" in df.columns else "number_of_unique_reviewers"

df["log_Files_Changed"] = np.log1p(pd.to_numeric(df["Files_Changed"], errors="coerce"))
df["log_Lines_Total"]   = np.log1p(pd.to_numeric(df["Lines_Total"], errors="coerce"))

df["SK"] = pd.to_numeric(df["SK"], errors="coerce")

need = [Y, "SK", "log_Files_Changed", "log_Lines_Total", "Repo"]
m = df[need].dropna()

baseline = choose_baseline(m)

formula = f"{Y} ~ SK + log_Files_Changed + log_Lines_Total + C(Repo, Treatment(reference='{baseline}'))"
nb = smf.glm(formula=formula, data=m, family=sm.families.NegativeBinomial()).fit(cov_type="HC3")

def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""

print("\n[Unique Reviewers — NB (log link)]")
print(f"N={int(nb.nobs)}, Pseudo-R^2={getattr(nb, 'prsquared', float('nan')):.3f}")
for term in ["SK", "log_Files_Changed", "log_Lines_Total"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")

# For any coefficient b on the log scale, % change = 100*(exp(b)-1).
print("\n[Interpretation as % change in expected # of reviewers]")
for term in ["SK", "log_Files_Changed", "log_Lines_Total"]:
    if term in nb.params.index:
        b, p = nb.params[term], nb.pvalues[term]
        pct = (np.exp(b) - 1.0) * 100.0
        label = {
            "SK": "SK (vs non-SK)",
            "log_Files_Changed": "Files_Changed (log1p)",
            "log_Lines_Total": "Lines_Total (log1p)"
        }[term]
        print(f"{label}: {pct:+.1f}%  (p={p:.4f}) {stars(p)}")



[Unique Reviewers — NB (log link)]
N=2274, Pseudo-R^2=nan
SK: 0.492 (p=0.0000) ***
log_Files_Changed: 0.095 (p=0.0000) ***
log_Lines_Total: 0.001 (p=0.8792) 

[Interpretation as % change in expected # of reviewers]
SK (vs non-SK): +63.5%  (p=0.0000) ***
Files_Changed (log1p): +10.0%  (p=0.0000) ***
Lines_Total (log1p): +0.1%  (p=0.8792) 


/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


## Total Discussion Comments

In [6]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = load_and_prepare()

Y = "total_discussion_comments" if "total_discussion_comments" in df.columns else "total_comments"

need = [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers", "Repo"]
m = df[need].copy()

for c in [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")

m = m.dropna(subset=need)

baseline = choose_baseline(m)

# Negative Binomial GLM
formula = f"{Y} ~ SK + Files_Changed + Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
nb = smf.glm(formula=formula, data=m, family=sm.families.NegativeBinomial()).fit(cov_type="HC3")

print("\n[Total Discussion Comments]")
print(f"N={int(nb.nobs)}")

# main predictors
print(f"SK: {nb.params['SK']:.3f} (p={nb.pvalues['SK']:.4f}) {stars(nb.pvalues['SK'])}")
for term in ["Files_Changed","Lines_Total","Reviewers"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")

for key in nb.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {nb.params[key]:.3f} (p={nb.pvalues[key]:.4f}) {stars(nb.pvalues[key])}")

print(f"(Baseline repo = {baseline})")



[Total Discussion Comments]
N=2274
SK: 0.746 (p=0.0000) ***
Files_Changed: 0.013 (p=0.0000) ***
Lines_Total: 0.000 (p=0.0250) *
Reviewers: 0.618 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: 0.577 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: 2.152 (p=0.0000) ***
(Baseline repo = AMReX)


/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [7]:
# log
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = load_and_prepare()

Y = "total_discussion_comments" if "total_discussion_comments" in df.columns else "total_comments"

# Log-transform predictors
df["log_Files_Changed"] = np.log1p(pd.to_numeric(df["Files_Changed"], errors="coerce"))
df["log_Lines_Total"]   = np.log1p(pd.to_numeric(df["Lines_Total"], errors="coerce"))
df["Reviewers"]         = pd.to_numeric(df["Reviewers"], errors="coerce")
df["SK"]                = pd.to_numeric(df["SK"], errors="coerce")

# Prepare data
need = [Y, "SK", "log_Files_Changed", "log_Lines_Total", "Reviewers", "Repo"]
m = df[need].dropna()

# Baseline repo
baseline = choose_baseline(m)

# Negative Binomial GLM with log link
formula = f"{Y} ~ SK + log_Files_Changed + log_Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
nb = smf.glm(formula=formula, data=m, family=sm.families.NegativeBinomial()).fit(cov_type="HC3")

def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""

print("\n[Total Discussion Comments — NB (log link)]")
print(f"N={int(nb.nobs)}")
for term in ["SK","log_Files_Changed","log_Lines_Total","Reviewers"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")

# For log-link NB, %Δ expected comments = 100*(exp(β)-1)
print("\n[Interpretation as % change in expected # of discussion comments]")
for term in ["SK","log_Files_Changed","log_Lines_Total","Reviewers"]:
    if term in nb.params.index:
        b, p = nb.params[term], nb.pvalues[term]
        pct = (np.exp(b) - 1) * 100
        label = {
            "SK": "SK (vs non-SK)",
            "log_Files_Changed": "Files_Changed (log1p)",
            "log_Lines_Total": "Lines_Total (log1p)",
            "Reviewers": "Reviewers (count)"
        }[term]
        print(f"{label}: {pct:+.1f}% change  (p={p:.4f}) {stars(p)}")

# Repo effects
for key in nb.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {nb.params[key]:.3f} (p={nb.pvalues[key]:.4f}) {stars(nb.pvalues[key])}")

print(f"(Baseline repo = {baseline})")



[Total Discussion Comments — NB (log link)]
N=2274
SK: 0.696 (p=0.0000) ***
log_Files_Changed: 0.188 (p=0.0000) ***
log_Lines_Total: 0.008 (p=0.5481) 
Reviewers: 0.609 (p=0.0000) ***

[Interpretation as % change in expected # of discussion comments]
SK (vs non-SK): +100.7% change  (p=0.0000) ***
Files_Changed (log1p): +20.7% change  (p=0.0000) ***
Lines_Total (log1p): +0.8% change  (p=0.5481) 
Reviewers (count): +83.9% change  (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: 0.528 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: 2.147 (p=0.0000) ***
(Baseline repo = AMReX)


/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


## Commits After First Review

In [8]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = load_and_prepare()
Y = "commits_after_review"

need = [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers", "Repo"]
m = df[need].copy()

# Numeric coercion for non-categorical fields
for c in [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")

# Drop rows with missing required fields
m = m.dropna(subset=need)

# Baseline repo
baseline = choose_baseline(m)

# Negative Binomial GLM with robust (HC3) SE
formula = f"{Y} ~ SK + Files_Changed + Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
nb = smf.glm(formula=formula, data=m, family=sm.families.NegativeBinomial()).fit(cov_type="HC3")

print("\n[Commits After First Review]")
print(f"N={int(nb.nobs)}")

# Main predictors
print(f"SK: {nb.params['SK']:.3f} (p={nb.pvalues['SK']:.4f}) {stars(nb.pvalues['SK'])}")
for term in ["Files_Changed", "Lines_Total", "Reviewers"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")

# Repo effects relative to baseline
for key in nb.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {nb.params[key]:.3f} (p={nb.pvalues[key]:.4f}) {stars(nb.pvalues[key])}")

print(f"(Baseline repo = {baseline})")




[Commits After First Review]
N=2274
SK: 1.463 (p=0.0000) ***
Files_Changed: 0.036 (p=0.0000) ***
Lines_Total: 0.000 (p=0.0000) ***
Reviewers: 0.655 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: 0.602 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: -0.197 (p=0.0448) *
(Baseline repo = AMReX)


/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [9]:
# log
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = load_and_prepare()
Y = "commits_after_review"

# Log-transform predictors
df["log_Files_Changed"] = np.log1p(pd.to_numeric(df["Files_Changed"], errors="coerce"))
df["log_Lines_Total"]   = np.log1p(pd.to_numeric(df["Lines_Total"], errors="coerce"))
df["Reviewers"]         = pd.to_numeric(df["Reviewers"], errors="coerce")
df["SK"]                = pd.to_numeric(df["SK"], errors="coerce")

# Prepare model frame
need = [Y, "SK", "log_Files_Changed", "log_Lines_Total", "Reviewers", "Repo"]
m = df[need].dropna()

# Baseline repo
baseline = choose_baseline(m)

# Negative Binomial GLM
formula = f"{Y} ~ SK + log_Files_Changed + log_Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
nb = smf.glm(formula=formula, data=m, family=sm.families.NegativeBinomial()).fit(cov_type="HC3")

def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""

print("\n[Commits After First Review — NB (log link)]")
print(f"N={int(nb.nobs)}")
for term in ["SK", "log_Files_Changed", "log_Lines_Total", "Reviewers"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")

# exp(β) − 1 → expected % change in commit count
print("\n[Interpretation as % change in expected # of commits after review]")
for term in ["SK", "log_Files_Changed", "log_Lines_Total", "Reviewers"]:
    if term in nb.params.index:
        b, p = nb.params[term], nb.pvalues[term]
        pct = (np.exp(b) - 1) * 100
        label = {
            "SK": "SK (vs non-SK)",
            "log_Files_Changed": "Files_Changed (log1p)",
            "log_Lines_Total": "Lines_Total (log1p)",
            "Reviewers": "Reviewers (count)"
        }[term]
        print(f"{label}: {pct:+.1f}% change  (p={p:.4f}) {stars(p)}")

for key in nb.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {nb.params[key]:.3f} (p={nb.pvalues[key]:.4f}) {stars(nb.pvalues[key])}")

print(f"(Baseline repo = {baseline})")



[Commits After First Review — NB (log link)]
N=2274
SK: 1.399 (p=0.0000) ***
log_Files_Changed: 0.556 (p=0.0000) ***
log_Lines_Total: 0.081 (p=0.0000) ***
Reviewers: 0.636 (p=0.0000) ***

[Interpretation as % change in expected # of commits after review]
SK (vs non-SK): +305.1% change  (p=0.0000) ***
Files_Changed (log1p): +74.3% change  (p=0.0000) ***
Lines_Total (log1p): +8.4% change  (p=0.0000) ***
Reviewers (count): +88.8% change  (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: 0.436 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: -0.224 (p=0.0166) *
(Baseline repo = AMReX)


/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


## Time Between First and Last Comments

In [10]:
import pandas as pd
import statsmodels.formula.api as smf

df = load_and_prepare()

# Convert to days
df["time_between_days"] = pd.to_numeric(df["time_between_first_last_comment"], errors="coerce") / 86400.0
Y = "time_between_days"

need = [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers", "Repo"]
m = df[need].copy()

# Numeric coercion
for c in [Y, "SK", "Files_Changed", "Lines_Total", "Reviewers"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")

m = m.dropna(subset=need)

baseline = choose_baseline(m)

formula = f"{Y} ~ SK + Files_Changed + Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
ols = smf.ols(formula=formula, data=m).fit(cov_type="HC3")

print("\n[Time Between First & Last Comments — in DAYS]")
print(f"N={int(ols.nobs)}, R^2={ols.rsquared:.3f}")

print(f"SK: {ols.params['SK']:.2f} days (p={ols.pvalues['SK']:.4f}) {stars(ols.pvalues['SK'])}")
for term in ["Files_Changed", "Lines_Total", "Reviewers"]:
    if term in ols.params.index:
        print(f"{term}: {ols.params[term]:.2f} days (p={ols.pvalues[term]:.4f}) {stars(ols.pvalues[term])}")

for key in ols.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {ols.params[key]:.2f} days (p={ols.pvalues[key]:.4f}) {stars(ols.pvalues[key])}")

print(f"(Baseline repo = {baseline})")



[Time Between First & Last Comments — in DAYS]
N=1590, R^2=0.045
SK: 5.32 days (p=0.0056) **
Files_Changed: 0.37 days (p=0.0022) **
Lines_Total: 0.00 days (p=0.6438) 
Reviewers: 5.54 days (p=0.0032) **
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: -0.93 days (p=0.7333) 
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: 2.94 days (p=0.2763) 
(Baseline repo = AMReX)


In [11]:
# log
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = load_and_prepare()

# Convert to days and log-transform
df["time_between_days"] = pd.to_numeric(df["time_between_first_last_comment"], errors="coerce") / 86400.0
df["log_time_between"]  = np.log1p(df["time_between_days"])

# Log-transform skewed predictors
df["log_Files_Changed"] = np.log1p(pd.to_numeric(df["Files_Changed"], errors="coerce"))
df["log_Lines_Total"]   = np.log1p(pd.to_numeric(df["Lines_Total"], errors="coerce"))
df["Reviewers"]         = pd.to_numeric(df["Reviewers"], errors="coerce")
df["SK"]                = pd.to_numeric(df["SK"], errors="coerce")

# Model frame
Y = "log_time_between"
need = [Y, "SK", "log_Files_Changed", "log_Lines_Total", "Reviewers", "Repo"]
m = df[need].dropna()

baseline = choose_baseline(m)

# Regression
formula = f"{Y} ~ SK + log_Files_Changed + log_Lines_Total + Reviewers + C(Repo, Treatment(reference='{baseline}'))"
ols = smf.ols(formula=formula, data=m).fit(cov_type="HC3")

def stars(p): return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""

print("\n[Time Between First & Last Comments — log(days) model]")
print(f"N={int(ols.nobs)}, R²={ols.rsquared:.3f}")
for term in ["SK", "log_Files_Changed", "log_Lines_Total", "Reviewers"]:
    if term in ols.params:
        print(f"{term}: {ols.params[term]:.3f} (p={ols.pvalues[term]:.4f}) {stars(ols.pvalues[term])}")

print("\n[Interpretation as % change in time between comments]")
for term in ["SK", "log_Files_Changed", "log_Lines_Total", "Reviewers"]:
    if term in ols.params:
        b, p = ols.params[term], ols.pvalues[term]
        pct = (np.expm1(b)) * 100
        label = {
            "SK": "SK (vs non-SK)",
            "log_Files_Changed": "Files_Changed (log1p)",
            "log_Lines_Total": "Lines_Total (log1p)",
            "Reviewers": "Reviewers (count)"
        }[term]
        print(f"{label}: {pct:+.1f}% change in duration  (p={p:.4f}) {stars(p)}")

for key in ols.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {ols.params[key]:.3f} (p={ols.pvalues[key]:.4f}) {stars(ols.pvalues[key])}")

print(f"(Baseline repo = {baseline})")



[Time Between First & Last Comments — log(days) model]
N=1590, R²=0.199
SK: 0.234 (p=0.0007) ***
log_Files_Changed: 0.373 (p=0.0000) ***
log_Lines_Total: -0.018 (p=0.2224) 
Reviewers: 0.339 (p=0.0000) ***

[Interpretation as % change in time between comments]
SK (vs non-SK): +26.4% change in duration  (p=0.0007) ***
Files_Changed (log1p): +45.2% change in duration  (p=0.0000) ***
Lines_Total (log1p): -1.8% change in duration  (p=0.2224) 
Reviewers (count): +40.3% change in duration  (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: 0.211 (p=0.0186) *
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: 0.442 (p=0.0000) ***
(Baseline repo = AMReX)


## Requested Changes

In [12]:
print("\n[Requested Changes]")
print(f"N={int(nb.nobs)}")

print(f"SK: {nb.params['SK']:.3f} (p={nb.pvalues['SK']:.4f}) {stars(nb.pvalues['SK'])}")
for term in ["Files_Changed", "Lines_Total", "Reviewers"]:
    if term in nb.params.index:
        print(f"{term}: {nb.params[term]:.3f} (p={nb.pvalues[term]:.4f}) {stars(nb.pvalues[term])}")

for key in nb.params.index:
    if key.startswith("C(Repo"):
        print(f"{key}: {nb.params[key]:.3f} (p={nb.pvalues[key]:.4f}) {stars(nb.pvalues[key])}")

print(f"(Baseline repo = {baseline})")



[Requested Changes]
N=2274
SK: 1.399 (p=0.0000) ***
Reviewers: 0.636 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: 0.436 (p=0.0000) ***
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: -0.224 (p=0.0166) *
(Baseline repo = AMReX)


In [13]:
# log
import numpy as np

print("\n[Requested Changes (log)]")
print(f"N={int(nb.nobs)}, Pseudo-R^2={getattr(nb, 'prsquared', float('nan')):.3f}")

def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

def show(term, label=None):
    if term in nb.params.index:
        b = nb.params[term]; p = nb.pvalues[term]
        irr = np.exp(b)
        pct = (irr - 1.0) * 100.0
        name = label or term
        print(f"{name}: β={b:.3f}, IRR={irr:.3f} ({pct:+.1f}%), p={p:.4f} {stars(p)}")

show("SK", "SK (vs non-SK)")
for term, label in [("Files_Changed","Files_Changed"),
                    ("Lines_Total","Lines_Total"),
                    ("Reviewers","Reviewers")]:
    show(term, label)

print("\nRepo effects (relative to baseline):")
for key in nb.params.index:
    if key.startswith("C(Repo"):
        show(key, key)

print(f"(Baseline repo = {baseline})")


[Requested Changes (log)]
N=2274, Pseudo-R^2=nan
SK (vs non-SK): β=1.399, IRR=4.051 (+305.1%), p=0.0000 ***
Reviewers: β=0.636, IRR=1.888 (+88.8%), p=0.0000 ***

Repo effects (relative to baseline):
C(Repo, Treatment(reference='AMReX'))[T.Mantid]: β=0.436, IRR=1.547 (+54.7%), p=0.0000 ***
C(Repo, Treatment(reference='AMReX'))[T.Trilinos]: β=-0.224, IRR=0.800 (-20.0%), p=0.0166 *
(Baseline repo = AMReX)
